# Teman Kopi — DECAFIA (CoffeeLeaf-CO) → Keras MobileNetV3Small

Run cells **1 → 5** on Google Colab (GPU: Runtime → Change runtime type → GPU).

Colab uses the runtime’s built-in TensorFlow — no pip install cell. **Stop after step 5.** Download `/content/teman_kopi_keras.keras`, then run TF.js export **locally** (see `training/README.md`).

## Before you start — put these files in Colab

| Required | Exact path |
| --- | --- |
| DECAFIA zip | `/content/decafia.zip` |
| Scripts | `/content/training/prepare_decafia.py` |
| | `/content/training/train.py` |

Upload via the left **Files** panel. This notebook does **not** download the dataset.
If a required file is missing, cells raise `FileNotFoundError`.

Zenodo: https://doi.org/10.5281/zenodo.19931903

Class remap: empty/`SANAS` → `healthy`; `minador`/`coco` → `pest_like`; `roya` → `disease_like`.

## 1. GPU check

Uses Colab’s preinstalled TensorFlow.

In [ ]:
import tensorflow as tf

print("tensorflow", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
if not gpus:
    print("WARNING: No GPU detected. Runtime → Change runtime type → GPU, then Restart session.")
else:
    print("GPU OK — TensorFlow will use it automatically.")

## 2. Assert required inputs exist

In [ ]:
from pathlib import Path

DECAFIA_ZIP = Path("/content/decafia.zip")
SCRIPTS_DIR = Path("/content/training")
REQUIRED = [
    DECAFIA_ZIP,
    SCRIPTS_DIR / "prepare_decafia.py",
    SCRIPTS_DIR / "train.py",
]

missing = [str(p) for p in REQUIRED if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "Missing required Colab files:\n  - "
        + "\n  - ".join(missing)
        + "\n\nUpload decafia.zip to /content/ and prepare_decafia.py + train.py to /content/training/."
    )

print("All required inputs found:")
for p in REQUIRED:
    print(" ", p, f"({p.stat().st_size} bytes)")

## 3. Unzip DECAFIA → `/content/decafia_raw`

In [ ]:
import shutil
from pathlib import Path

DECAFIA_ZIP = Path("/content/decafia.zip")
RAW_DIR = Path("/content/decafia_raw")

if not DECAFIA_ZIP.is_file():
    raise FileNotFoundError(f"Missing {DECAFIA_ZIP}")

if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)
RAW_DIR.mkdir(parents=True, exist_ok=True)

!unzip -q /content/decafia.zip -d /content/decafia_raw

files = [p for p in RAW_DIR.rglob("*") if p.is_file() and "__MACOSX" not in str(p)]
if not files:
    raise FileNotFoundError(f"{RAW_DIR} is empty after unzip. Is decafia.zip valid?")

print(f"Unzipped OK — {len(files)} files under {RAW_DIR}")
!find /content/decafia_raw -maxdepth 3 -type d | head -40

## 4. Remap YOLO → classification folders

`healthy` / `pest_like` / `disease_like` → `/content/decafia_remapped`

In [ ]:
from pathlib import Path

PREPARE = Path("/content/training/prepare_decafia.py")
RAW_DIR = Path("/content/decafia_raw")
REMAPPED = Path("/content/decafia_remapped")

if not PREPARE.is_file():
    raise FileNotFoundError(f"Missing {PREPARE}")
if not RAW_DIR.exists():
    raise FileNotFoundError(f"Missing {RAW_DIR} — run the unzip cell first.")

!python /content/training/prepare_decafia.py --source /content/decafia_raw --out /content/decafia_remapped

classes = ["healthy", "pest_like", "disease_like"]
exts = {".jpg", ".jpeg", ".png"}
counts = {}
for split in ("train", "val"):
    for cls in classes:
        d = REMAPPED / split / cls
        if not d.is_dir():
            raise FileNotFoundError(f"Expected remapped folder missing: {d}")
        n = sum(1 for p in d.iterdir() if p.suffix.lower() in exts)
        counts[f"{split}/{cls}"] = n

total = sum(counts.values())
if total == 0:
    raise FileNotFoundError("Remap produced 0 images.")

print("Remap counts:")
for k, v in counts.items():
    print(f"  {k}: {v}")
print("Total images:", total)

## 5. Train MobileNetV3Small

Writes `/content/teman_kopi_keras.keras` — **last Colab step**. Download that file from the Files panel, then export TF.js on your laptop.

In [ ]:
from pathlib import Path

TRAIN = Path("/content/training/train.py")
DATA = Path("/content/decafia_remapped")
KERAS_OUT = Path("/content/teman_kopi_keras.keras")

if not TRAIN.is_file():
    raise FileNotFoundError(f"Missing {TRAIN}")
if not (DATA / "train").is_dir():
    raise FileNotFoundError(f"Missing {DATA / 'train'} — run remap cell first.")

!python /content/training/train.py --data-dir /content/decafia_remapped --epochs 8 --out /content/teman_kopi_keras.keras

if not KERAS_OUT.is_file():
    raise FileNotFoundError(f"Training finished but model not found: {KERAS_OUT}")
print("Keras model saved:", KERAS_OUT, KERAS_OUT.stat().st_size, "bytes")
print()
print("Colab done. Download this file:")
print(" ", KERAS_OUT)
print("Then locally (Python 3.11 or 3.12 venv — see training/README.md):")
print("  python training/export_tfjs.py --model ~/Downloads/teman_kopi_keras.keras --out public/models/teman-kopi")

## Next (local only) — TF.js export

Do **not** continue export on Colab. On your machine:

1. Download `/content/teman_kopi_keras.keras`
2. Use **Python 3.11 or 3.12** (not 3.14 — TensorFlow has no wheels yet)
3. Create a venv and install export deps:

```bash
python3.11 -m venv .venv-train
source .venv-train/bin/activate
pip install "tensorflow>=2.15,<2.20" "tensorflowjs==4.22.0"
# If tensorflowjs fails with np.object / NumPy errors:
# pip install "numpy<2"
```

4. Export into the PWA model folder:

```bash
python training/export_tfjs.py \
  --model ~/path/to/file.keras \
  --out public/models/teman-kopi
```

5. Confirm `public/models/teman-kopi/model.json` exists, then `npm run build` (push to `main` to deploy).

**Time on Mac (typical):** first-time TF/tfjs install ~5–15+ min; export itself ~1–5 min; verifying `model.json` is seconds.